# Binôme B — Évaluation finale du modèle figé

**Phase CRISP-DM : Evaluation.** Le modèle, la perte, σ et le seuil sont chargés depuis `models/B_best_config.json`, produit par le notebook Modeling. Cette évaluation ne modifie aucun paramètre.

> **Test consulté lors d'une itération antérieure ; résultats exploratoires ; aucune décision modèle/post-traitement/XAI ne doit en découler.** Les résultats test ne constituent donc pas une estimation finale indépendante, même si ce notebook les rapporte pour transparence. Toute sélection et tout XAI restent limités à train/validation.

## 1. Configuration figée et protocole

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Racine du dépôt introuvable.")
sys.path.insert(0, str(ROOT / "src"))
from roadanomaly import data_pipeline as dp, metrics
from roadanomaly.anomaly_detector import AnomalyDetector

IMG_SIZE = (352, 640)
MIN_OBJECT_PX = 4
SMALL_OBJECT_PX = round(1000 * (352 / 720) ** 2)
FPR_LIMIT = 0.3
SEED = 1337
CONFIG_PATH = ROOT / "models" / "B_best_config.json"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError("Exécuter d'abord notebooks/02_B_Model_Improvement.ipynb pour créer B_best_config.json.")
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
detector = AnomalyDetector(CONFIG_PATH)
print("Modèle:", config["model_path"])
print("Loss:", config["training"]["loss"], "| sigma:", detector.sigma, "| seuil:", detector.threshold)
print("Configuration du gagnant validation chargée ; aucun réglage ne sera modifié.")

## 2. Évaluation validation et baselines triviales

In [ ]:
val_frames, x_val, y_val = dp.load_split("val", IMG_SIZE)
gt_val = y_val[..., 0].astype(bool)
val_predictions = [detector.predict(image) for image in x_val]
val_scores = [item["score_map"] for item in val_predictions]
val_masks = [item["mask"] for item in val_predictions]
val_auprc = average_precision_score(
    np.concatenate([g.ravel() for g in gt_val]),
    np.concatenate([s.ravel() for s in val_scores]),
)
val_auroc = roc_auc_score(
    np.concatenate([g.ravel() for g in gt_val]),
    np.concatenate([s.ravel() for s in val_scores]),
)
val_aupro = metrics.aupro(list(gt_val), val_scores, MIN_OBJECT_PX, fpr_limit=FPR_LIMIT)
val_pixel = metrics.evaluate_masks(list(gt_val), val_masks)
val_objects = [metrics.object_scores(g,p,MIN_OBJECT_PX) for g,p in zip(gt_val,val_masks)]
val_sizes = [metrics.object_recall_by_size(g,p,MIN_OBJECT_PX,SMALL_OBJECT_PX) for g,p in zip(gt_val,val_masks)]
val_summary = {
    "AuPRC": float(val_auprc), "AuPRC_chance_prevalence": float(np.mean([m.mean() for m in gt_val])),
    "AUROC": float(val_auroc), "AUROC_chance": 0.5,
    "AUPRO@0.3": float(val_aupro), "AUPRO_chance": 0.15,
    "pixel_per_image_mean": val_pixel["per_image_mean"],
    "pixel_aggregated": val_pixel["aggregated"],
    "objects": val_objects, "size_counts": val_sizes,
}
for name, pred in [("tout fond", [np.zeros_like(g) for g in gt_val]), ("tout anomalie", [np.ones_like(g) for g in gt_val])]:
    print(name, metrics.evaluate_masks(list(gt_val), pred)["per_image_mean"])
print(json.dumps(val_summary, indent=2))

## 3. Diagnostic XAI — validation uniquement

La heatmap Grad-CAM vise la MSE brute moyenne à l'intérieur de la région prédite en validation. Si aucune région n'est prédite, le notebook utilise le centile supérieur des scores de cette image. La vérité terrain est affichée uniquement comme contexte. Cette attribution est un diagnostic et **ne constitue pas une explication causale**.

In [ ]:
xai_index = int(np.argmax([mask.sum() for mask in val_masks]))
xai_image = x_val[xai_index]
xai_gt = gt_val[xai_index]
xai_prediction = val_predictions[xai_index]
xai_heatmap = detector.explain(xai_image)
xai_reconstruction = detector.model(xai_image[None, ...], training=False).numpy()[0]
xai_raw_error = np.mean(np.square(xai_image - xai_reconstruction), axis=-1)
assert xai_heatmap.shape == IMG_SIZE and np.isfinite(xai_heatmap).all()
fig, axes = plt.subplots(1, 6, figsize=(20, 4), tight_layout=True)
axes[0].imshow(xai_image); axes[0].set_title("Image")
axes[1].imshow(xai_gt, cmap="gray"); axes[1].set_title("GT (contexte)")
axes[2].imshow(xai_reconstruction); axes[2].set_title("Reconstruction")
axes[3].imshow(xai_raw_error, cmap="magma"); axes[3].set_title("MSE brute")
axes[4].imshow(xai_image); axes[4].imshow(xai_heatmap, cmap="jet", alpha=0.45); axes[4].set_title("Grad-CAM du code")
axes[5].imshow(xai_prediction["mask"], cmap="gray"); axes[5].set_title("Région prédite")
for axis in axes: axis.axis("off")
plt.savefig(ROOT / "reports" / "figures" / "B_03_validation_gradcam.png", dpi=120, bbox_inches="tight")
plt.show()

def region_mse(image, region):
    reconstruction = detector.model(image[None,...], training=False).numpy()[0]
    error = np.mean(np.square(image-reconstruction),axis=-1)
    return float(error[region].mean())
region = xai_prediction["mask"].astype(bool)
if not region.any(): region = xai_prediction["score_map"] >= np.quantile(xai_prediction["score_map"],0.99)
coords = np.argwhere(region)
ranked = coords[np.argsort(xai_heatmap[region])[::-1]]
neutral = xai_image.mean(axis=(0,1))
fractions = np.linspace(0,1,6)
delete_values, insert_values = [], []
for fraction in fractions:
    count = int(round(fraction*len(ranked)))
    selected = ranked[:count]
    deleted = xai_image.copy()
    inserted = np.broadcast_to(neutral,xai_image.shape).copy()
    if count:
        deleted[selected[:,0],selected[:,1]]=neutral
        inserted[selected[:,0],selected[:,1]]=xai_image[selected[:,0],selected[:,1]]
    delete_values.append(region_mse(deleted,region))
    insert_values.append(region_mse(inserted,region))
plt.figure(figsize=(6,4))
plt.plot(fractions,delete_values,marker="o",label="Suppression")
plt.plot(fractions,insert_values,marker="o",label="Insertion")
plt.xlabel("Fraction des pixels de la ROI perturbés"); plt.ylabel("MSE brute dans la ROI")
plt.title("Sanity check XAI — validation seulement"); plt.legend(); plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "B_04_validation_xai_perturbation.png",dpi=120,bbox_inches="tight")
plt.show()
print("La suppression/insertion est un sanity check, pas une preuve de causalité.")

## 4. Résultats test exploratoires — sans ajustement

> **Test consulté lors d'une itération antérieure ; résultats exploratoires ; aucune décision modèle/post-traitement/XAI ne doit en découler.** Aucun choix ne sera révisé après cette cellule. Le test actuel ne peut pas être présenté comme un holdout indépendant.

In [ ]:
test_frames, x_test, y_test = dp.load_split("test", IMG_SIZE)
gt_test = y_test[...,0].astype(bool)
test_predictions = [detector.predict(image) for image in x_test]
test_scores = [item["score_map"] for item in test_predictions]
test_masks = [item["mask"] for item in test_predictions]
test_auprc = average_precision_score(
    np.concatenate([g.ravel() for g in gt_test]), np.concatenate([s.ravel() for s in test_scores])
)
test_auroc = roc_auc_score(
    np.concatenate([g.ravel() for g in gt_test]), np.concatenate([s.ravel() for s in test_scores])
)
test_aupro = metrics.aupro(list(gt_test),test_scores,MIN_OBJECT_PX,fpr_limit=FPR_LIMIT)
test_pixel = metrics.evaluate_masks(list(gt_test),test_masks)
test_objects = [metrics.object_scores(g,p,MIN_OBJECT_PX) for g,p in zip(gt_test,test_masks)]
test_sizes = [metrics.object_recall_by_size(g,p,MIN_OBJECT_PX,SMALL_OBJECT_PX) for g,p in zip(gt_test,test_masks)]
assert len(test_frames)==9, f"Le protocole attendu a 9 images test, reçu {len(test_frames)}"
bootstrap = {}
for metric_name in ("precision","recall","f1","iou","dice"):
    values = [row[metric_name] for row in test_pixel["per_image"]]
    mean, low, high = metrics.bootstrap_ci(values,n_boot=2000,alpha=0.05,seed=SEED)
    bootstrap[metric_name] = {"mean":mean,"ci95":[low,high]}
test_summary = {
    "AuPRC":float(test_auprc),"AuPRC_chance_prevalence":float(np.mean([g.mean() for g in gt_test])),
    "AUROC":float(test_auroc),"AUROC_chance":0.5,
    "AUPRO@0.3":float(test_aupro),"AUPRO_chance":0.15,
    "pixel_per_image_mean":test_pixel["per_image_mean"],"pixel_aggregated":test_pixel["aggregated"],
    "bootstrap_95_percent_images":bootstrap,"objects":test_objects,"size_counts":test_sizes,
}
print("TEST EXPLORATOIRE — ne pas réutiliser pour sélectionner ou ajuster le modèle")
print(json.dumps(test_summary,indent=2))

## 5. Tableau par image et analyse d'erreurs

In [ ]:
per_image_rows=[]
for i,(frame,truth,prediction,pixel,objects) in enumerate(zip(
    test_frames,gt_test,test_masks,test_pixel["per_image"],test_objects
)):
    per_image_rows.append({
        "frame":frame,"precision":pixel["precision"],"recall":pixel["recall"],"F1":pixel["f1"],
        "IoU":pixel["iou"],"Dice":pixel["dice"],"anomaly_percent":100*float(truth.mean()),
        "objects_detected":objects["detected"],"objects_total":objects["gt_objects"],
        "false_positive_blobs":objects["false_alarms"],
    })
per_image_table=pd.DataFrame(per_image_rows)
print(per_image_table.round(4).to_string(index=False))
order=np.argsort([-row["f1"] for row in test_pixel["per_image"]])
fig,axes=plt.subplots(len(order),5,figsize=(20,2.7*len(order)),squeeze=False,tight_layout=True)
for row_index,image_index in enumerate(order):
    pred=test_masks[image_index].astype(bool)
    errors=np.zeros((*IMG_SIZE,3),dtype=np.float32)
    errors[pred & gt_test[image_index]]=(1,1,1)
    errors[pred & ~gt_test[image_index]]=(1,0,0)
    errors[~pred & gt_test[image_index]]=(0,0.4,1)
    reconstruction=detector.model(x_test[image_index:image_index+1],training=False).numpy()[0]
    panels=[(x_test[image_index],test_frames[image_index]),
            (gt_test[image_index],"GT"),(reconstruction,"Reconstruction"),
            (test_scores[image_index],"Score z"),(errors,f"TP blanc / FP rouge / FN bleu — F1={test_pixel['per_image'][image_index]['f1']:.3f}")]
    for col,(panel,title) in enumerate(panels):
        axes[row_index,col].imshow(panel,cmap="magma" if col==3 else ("gray" if panel.ndim==2 else None))
        axes[row_index,col].set_title(title[:45],fontsize=8); axes[row_index,col].axis("off")
plt.savefig(ROOT / "reports" / "figures" / "B_05_per_image_grid.png",dpi=80,bbox_inches="tight")
plt.show()

## 6. Export des résultats figés

In [ ]:
final_report = {
    "warning": "Test consulté lors d'une itération antérieure ; résultats exploratoires ; aucune décision modèle/post-traitement/XAI ne doit en découler.",
    "model_config": config,
    "validation": val_summary,
    "test_exploratory": test_summary,
    "test_per_image": per_image_rows,
    "xai": {
        "split":"validation",
        "target":"MSE brute moyenne dans la région prédite",
        "causal_explanation":False,
        "deletion_scores":delete_values,
        "insertion_scores":insert_values,
    },
    "figures":[
        "reports/figures/B_01_learning_curves.png",
        "reports/figures/B_02_sigma_and_threshold.png",
        "reports/figures/B_03_validation_gradcam.png",
        "reports/figures/B_04_validation_xai_perturbation.png",
        "reports/figures/B_05_per_image_grid.png",
    ],
}
report_path=ROOT / "reports" / "B_final_report.json"
report_path.write_text(json.dumps(final_report,indent=2),encoding="utf-8")
print("Rapport écrit :",report_path)
print("B_01 et B_02 proviennent du notebook Modeling ; les trois figures restantes viennent de ce notebook.")